# Phase 2: Condition Comparison (Normal vs Muscimol)

Interactive dashboard for comparing perturbation responses between normal and muscimol conditions.

**Features:**
- Load same animal in two conditions (normal and muscimol)
- Compare responses per body part
- Analyze latencies, magnitudes, and recovery
- Track limb asymmetry
- View overlaid 2D trajectories

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../")

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 2**128
from ipywidgets import (
    interact, interactive, fixed, IntSlider, Dropdown, Button, HBox, VBox,
    Label, Output, FloatSlider
)
from IPython.display import display, clear_output, HTML

from tools.behavior import (
    BehaviorDataset, TimeSeriesPlotter, load_session_registry, SessionComparator
)

print("✓ Imports successful")

✓ Imports successful


## Load Session Registry and Find Session Pairs

In [2]:
# Load registry
registry = load_session_registry()

# Build session triplets (normal day 1, day 2, muscimol for each animal)
def build_session_triplets():
    """Find matching normal (day 1, 2) and muscimol sessions for each animal."""
    triplets = {}
    
    # Group by animal
    by_animal = {}
    for session in registry.sessions:
        if session.animal_id not in by_animal:
            by_animal[session.animal_id] = {}
        if session.condition not in by_animal[session.animal_id]:
            by_animal[session.animal_id][session.condition] = []
        by_animal[session.animal_id][session.condition].append(session)
    
    # Create triplets (normal day 1 + day 2 + muscimol)
    for animal_id, conditions in by_animal.items():
        if 'normal' in conditions and 'muscimol' in conditions:
            # Get normal sessions sorted by day
            normal_sessions = sorted(conditions['normal'], key=lambda s: s.day_num)
            
            # We need at least 2 normal sessions
            if len(normal_sessions) >= 2:
                day1 = normal_sessions[0]
                day2 = normal_sessions[1]
                # Use first muscimol session for this animal
                muscimol = conditions['muscimol'][0]
                
                triplet_name = f"{animal_id}: Day1 ({day1.short_label}) vs Day2 ({day2.short_label}) vs Muscimol ({muscimol.short_label})"
                triplets[triplet_name] = (day1.session_name, day2.session_name, muscimol.session_name)
    
    return triplets

session_triplets = build_session_triplets()
print(f"✓ Found {len(session_triplets)} animal triplets with normal (day 1 & 2) and muscimol:")
for triplet_name in session_triplets.keys():
    print(f"  {triplet_name}")


✓ Found 4 animal triplets with normal (day 1 & 2) and muscimol:
  M078: Day1 (M078-Norm-D1) vs Day2 (M078-Norm-D2) vs Muscimol (M078-Musc-D4)
  M086: Day1 (M086-Norm-D1) vs Day2 (M086-Norm-D2) vs Muscimol (M086-Musc-D3)
  M103: Day1 (M103-Norm-D1) vs Day2 (M103-Norm-D2) vs Muscimol (M103-Musc-D3)
  M106: Day1 (M106-Norm-D1) vs Day2 (M106-Norm-D2) vs Muscimol (M106-Musc-D3)


## Initialize Controls

In [3]:
# Session triplet selector
pair_dropdown = Dropdown(
    options=list(session_triplets.keys()),
    value=list(session_triplets.keys())[0] if session_triplets else None,
    description='Animal Triplet:',
    layout={'width': '500px'}
)

# Direction slider
direction_slider = IntSlider(
    value=0,
    min=0,
    max=11,
    step=1,
    description='Direction:',
    layout={'width': '400px'}
)

# Body part selector
bodypart_dropdown = Dropdown(
    options=['all'] + ['left_foot', 'right_foot', 'hip_center', 'shoulder_center', 'left_paw', 'right_paw'],
    value='all',
    description='Body Part:',
    layout={'width': '300px'}
)

# Time window controls
pre_ms_slider = IntSlider(
    value=200,
    min=50,
    max=500,
    step=50,
    description='Pre (ms):',
    layout={'width': '400px'}
)

post_ms_slider = IntSlider(
    value=1500,
    min=500,
    max=3000,
    step=100,
    description='Post (ms):',
    layout={'width': '400px'}
)

print("✓ Controls created")


✓ Controls created


## Comparison Functions

In [4]:
# Global state
current_comparator = None
current_triplet_name = None

def load_comparison(triplet_name):
    """Load a session triplet for comparison."""
    global current_comparator, current_triplet_name
    
    if triplet_name == current_triplet_name and current_comparator is not None:
        return
    
    day1_session, day2_session, muscimol_session = session_triplets[triplet_name]
    print(f"Loading {triplet_name}...")
    current_comparator = SessionComparator(day1_session, day2_session, muscimol_session)
    current_triplet_name = triplet_name
    print(f"✓ Loaded comparison: {day1_session} vs {day2_session} vs {muscimol_session}")

def on_triplet_change(change):
    """Callback when session triplet changes."""
    load_comparison(change['new'])

pair_dropdown.observe(on_triplet_change, names='value')

# Load initial triplet
if session_triplets:
    load_comparison(list(session_triplets.keys())[2])
    print("\n✓ Functions defined")


Loading M103: Day1 (M103-Norm-D1) vs Day2 (M103-Norm-D2) vs Muscimol (M103-Musc-D3)...
M103_2026_02_18_15_30
fields: ['values_before_camera_trigger', 'idx_before_camera_trigger', 'values_Sol_duration', 'values_Sol_direction'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'idx_Sol_duration', 'values_Sol_direction', 'idx_Sol_direction', 'idx_sol_on'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'values_Sol_direction'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'idx_Sol_duration', 'values_Sol_direction', 'idx_Sol_direction', 'idx_sol_on'] could not be converted to int.
Repairing columns ['MotSen1_X', 'MotSen1_Y']
Extending index to 41999 in trial: free and id: 996, inserting NaN.
Extending index to 41999 in trial: free and id: 996, inserting NaN.
Combined every 1 bins
Resulting all_spikes ephys data shape is (NxT): (15, 48000)
Resulting CP_

## Metrics Comparison Table

In [5]:
def show_metrics_table(direction, pre_ms, post_ms):
    """Display metrics comparison table across all three conditions."""
    if current_comparator is None:
        print("⚠ No active comparison")
        return
    
    try:
        print(f"Computing metrics for direction {direction}...")
        df = current_comparator.compare_conditions(direction, 'trial', pre_ms, post_ms)
        
        # Filter if specific body part selected
        if bodypart_dropdown.value != 'all':
            df = df[df['Body Part'].str.lower() == bodypart_dropdown.value.replace('_', ' ')]
        
        # Display formatted table
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        
        print(f"\n{'='*120}")
        print(f"Animal: {current_comparator.animal_id} | Direction: {direction}")
        print(f"{'='*120}")
        display(df.round(2))
        
    except Exception as e:
        print(f"✗ Error: {str(e)[:200]}")
        import traceback
        traceback.print_exc()

print("Creating metrics plotter...")
out_metrics = interactive(show_metrics_table,
                          direction=direction_slider,
                          pre_ms=pre_ms_slider,
                          post_ms=post_ms_slider)

display(VBox([
    Label("Metrics Comparison (Normal Day 1 vs Day 2 vs Muscimol)"),
    HBox([pair_dropdown]),
    HBox([bodypart_dropdown, direction_slider]),
    HBox([pre_ms_slider]),
    HBox([post_ms_slider]),
    out_metrics]))


Creating metrics plotter...


## Session Pair Info

In [10]:
def plot_scatter_comparison(direction, metric_name, pre_ms, post_ms):
    """Create scatter plots comparing trial-level metrics across 3 conditions."""
    if current_comparator is None:
        print("⚠ No active comparison")
        return
    
    try:
        # Get trial-level metrics
        trial_data = current_comparator.get_trial_level_metrics(direction, 'trial', pre_ms, post_ms)
        
        # Get a body part (default first one, or could make it selectable)
        body_part = list(trial_data['Normal Day 1'].keys())[0]
        
        # Create figure with 3 subplots (one per metric type)
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle(f"Direction {direction} | Body Part: {body_part.replace('_', ' ').title()}", 
                     fontsize=14, fontweight='bold')
        
        # Prepare data for metric
        conditions = ['Normal Day 1', 'Normal Day 2', 'Muscimol']
        colors = ['steelblue', 'darkgreen', 'darkorange']
        x_positions = [0, 1.5, 3]
        
        metrics_to_plot = ['peak_latency_ms', 'peak_magnitude_cm', 'recovery_latency_ms']
        
        for ax_idx, metric in enumerate(metrics_to_plot):
            ax = axes[ax_idx]
            
            # Plot scatter and mean/SD for each condition
            for cond_idx, condition in enumerate(conditions):
                # Get trial values
                values = trial_data[condition][body_part].get(metric, [])
                values = [v for v in values if v is not None]  # Remove None values
                
                if not values:
                    continue
                
                # Scatter plot (with jitter)
                x = np.random.normal(x_positions[cond_idx], 0.04, len(values))
                ax.scatter(x, values, alpha=0.6, s=50, color=colors[cond_idx], 
                          label=condition, edgecolors='black', linewidth=0.5)
                
                # Mean line
                mean_val = np.mean(values)
                sd_val = np.std(values)
                ax.hlines(mean_val, x_positions[cond_idx] - 0.2, x_positions[cond_idx] + 0.2, 
                         colors=colors[cond_idx], linewidth=3, linestyles='solid')
                
                # SD bands
                ax.fill_between([x_positions[cond_idx] - 0.2, x_positions[cond_idx] + 0.2],
                               mean_val - sd_val, mean_val + sd_val,
                               alpha=0.2, color=colors[cond_idx])
                
                # Add text annotation
                ax.text(x_positions[cond_idx], mean_val + sd_val + (0.1 * (max(values) - min(values))),
                       f'μ={mean_val:.1f}\nσ={sd_val:.1f}\nn={len(values)}',
                       ha='center', fontsize=9, bbox=dict(boxstyle='round', 
                       facecolor=colors[cond_idx], alpha=0.2))
            
            # Format subplot
            ax.set_xlim(-0.5, 3.5)
            ax.set_xticks(x_positions)
            ax.set_xticklabels(conditions, rotation=15, ha='right')
            ax.set_ylabel(metric.replace('_', ' '), fontsize=11)
            ax.grid(axis='y', alpha=0.3)
            ax.set_title(metric.replace('_', ' ').title(), fontsize=12)
        
        axes[0].legend(loc='upper left', fontsize=9)
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"✗ Error: {str(e)}")
        import traceback
        traceback.print_exc()

# Create interactive scatter plot
print("Creating scatter plot comparator...")
scatter_metric = Dropdown(
    options=['peak_latency_ms', 'peak_magnitude_cm', 'recovery_latency_ms', 'correction_latency_ms'],
    value='peak_latency_ms',
    description='Metric:',
    layout={'width': '300px'}
)

out_scatter = interactive(plot_scatter_comparison,
                         direction=direction_slider,
                         metric_name=fixed('peak_latency_ms'),  # Not used in function, but kept for interface
                         pre_ms=pre_ms_slider,
                         post_ms=post_ms_slider)

display(VBox([
    Label("Trial-Level Scatter Comparison"),
    HBox([pair_dropdown]),
    HBox([direction_slider, scatter_metric]),
    HBox([pre_ms_slider, post_ms_slider]),
    out_scatter
]))

Creating scatter plot comparator...


In [13]:
def plot_scatter_single_metric(direction, bodypart, metric_name, pre_ms, post_ms, use_manual):
    """Create a detailed scatter plot for a single metric and body part."""
    if current_comparator is None:
        print("⚠ No active comparison")
        return
    
    try:
        # Get trial-level metrics (auto or manual)
        if use_manual and len(manual_annotations) > 0:
            trial_data = current_comparator.get_trial_level_metrics(
                direction, 'trial', pre_ms, post_ms,
                manual_annotations=manual_annotations,
                use_manual_annotations=True
            )
            data_source = "MANUAL ANNOTATIONS"
        else:
            trial_data = current_comparator.get_trial_level_metrics(
                direction, 'trial', pre_ms, post_ms,
                use_manual_annotations=False
            )
            data_source = "AUTO-DETECTED"
        
        # Filter body part
        if bodypart != 'all':
            selected_body_part = bodypart
        else:
            if 'Normal Day 1' in trial_data and len(trial_data['Normal Day 1']) > 0:
                selected_body_part = list(trial_data['Normal Day 1'].keys())[0]
            else:
                print("⚠ No data available for this selection")
                return
        
        # Create figure
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Prepare data for metric
        conditions = ['Normal Day 1', 'Normal Day 2', 'Muscimol']
        colors = ['steelblue', 'darkgreen', 'darkorange']
        x_positions = [0, 1.5, 3]
        
        all_values = []
        stats_text = []
        
        # Plot scatter and mean/SD for each condition
        for cond_idx, condition in enumerate(conditions):
            # Check if condition has data
            if condition not in trial_data or selected_body_part not in trial_data[condition]:
                continue
            
            # Get trial values
            values = trial_data[condition][selected_body_part].get(metric_name, [])
            values = [v for v in values if v is not None]  # Remove None values
            all_values.extend(values)
            
            if not values:
                continue
            
            # Scatter plot (with jitter)
            x = np.random.normal(x_positions[cond_idx], 0.05, len(values))
            ax.scatter(x, values, alpha=0.6, s=80, color=colors[cond_idx], 
                      label=condition, edgecolors='black', linewidth=1)
            
            # Statistics
            mean_val = np.mean(values)
            sd_val = np.std(values)
            sem_val = sd_val / np.sqrt(len(values))
            
            # Mean line
            ax.hlines(mean_val, x_positions[cond_idx] - 0.25, x_positions[cond_idx] + 0.25, 
                     colors=colors[cond_idx], linewidth=4, linestyles='solid', zorder=10)
            
            # SD bands
            ax.fill_between([x_positions[cond_idx] - 0.25, x_positions[cond_idx] + 0.25],
                           mean_val - sd_val, mean_val + sd_val,
                           alpha=0.15, color=colors[cond_idx], zorder=5)
            
            # SEM error bars
            ax.errorbar([x_positions[cond_idx]], [mean_val], yerr=[sem_val],
                       fmt='none', ecolor=colors[cond_idx], elinewidth=2, capsize=8, zorder=9)
            
            # Add text annotation
            stats_text.append(f"{condition}:\n μ={mean_val:.2f}\n σ={sd_val:.2f}\n SEM={sem_val:.2f}\n n={len(values)}")
        
        # Format plot
        ax.set_xlim(-0.7, 3.7)
        ax.set_xticks(x_positions)
        ax.set_xticklabels(conditions, fontsize=11)
        ax.set_ylabel(metric_name.replace('_', ' ').title(), fontsize=12, fontweight='bold')
        ax.set_title(f"Direction {direction} | Body Part: {selected_body_part.replace('_', ' ').upper()} | Source: {data_source}", 
                    fontsize=13, fontweight='bold')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
        
        # Add statistics box
        stats_box = '\n'.join(stats_text) if stats_text else "No data"
        ax.text(0.98, 0.97, stats_box, transform=ax.transAxes, fontsize=9,
               verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
               family='monospace')
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"✗ Error: {str(e)}")
        import traceback
        traceback.print_exc()

# Create metric selector dropdown
metric_dropdown = Dropdown(
    options=['peak_latency_ms', 
            'peak_magnitude_cm', 
            'recovery_latency_ms', 
            'correction_latency_ms', 
            'peak_x_cm', 'peak_y_cm', 'peak_z_cm'],
    value='peak_latency_ms',
    description='Metric:',
    layout={'width': '300px'}
)

In [ ]:
# Create a checkbox toggle for auto vs manual
from ipywidgets import Checkbox

use_manual_annotations_checkbox = Checkbox(
    value=False,
    description='Use Manual Annotations Only',
    indent=False
)

# Update the scatter plot to use this checkbox
out_scatter_single = interactive(plot_scatter_single_metric,
                                direction=direction_slider,
                                bodypart=bodypart_dropdown,
                                metric_name=metric_dropdown,
                                pre_ms=pre_ms_slider,
                                post_ms=post_ms_slider,
                                use_manual=use_manual_annotations_checkbox)

display(VBox([
    Label("Single Metric Scatter Comparison (with μ, σ, SEM)"),
    HBox([pair_dropdown]),
    HBox([direction_slider, bodypart_dropdown]),
    HBox([metric_dropdown]),
    HBox([pre_ms_slider, post_ms_slider]),
    use_manual_annotations_checkbox,
    Label("💡 Enable to scatter only manually-annotated trials. Requires saved annotations."),
    out_scatter_single
]))

## Trial-Level Scatter Plots

Visualize individual trial values with mean and standard deviation overlays for comparing Normal Day 1, Day 2, and Muscimol responses.

In [7]:
def show_pair_info():
    """Display information about current session triplet."""
    if current_comparator is None:
        print("No comparison loaded")
        return
    
    print(f"\n{'='*80}")
    print(f"COMPARISON TRIPLET: {current_triplet_name}")
    print(f"{'='*80}")
    
    print(f"\nNORMAL DAY 1 ({current_comparator.day1_name}):")
    day1 = current_comparator.day1_dataset
    print(f"  Directions: {day1.directions}")
    print(f"  Trials per direction: {day1.n_trials_per_direction}")
    
    print(f"\nNORMAL DAY 2 ({current_comparator.day2_name}):")
    day2 = current_comparator.day2_dataset
    print(f"  Directions: {day2.directions}")
    print(f"  Trials per direction: {day2.n_trials_per_direction}")
    
    print(f"\nMUSCIMOL ({current_comparator.muscimol_name}):")
    muscimol = current_comparator.muscimol_dataset
    print(f"  Directions: {muscimol.directions}")
    print(f"  Trials per direction: {muscimol.n_trials_per_direction}")
    
    print(f"\n{'='*80}\n")

show_pair_info()



COMPARISON TRIPLET: M103: Day1 (M103-Norm-D1) vs Day2 (M103-Norm-D2) vs Muscimol (M103-Musc-D3)

NORMAL DAY 1 (M103_2026_02_18_15_30):
  Directions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
  Trials per direction: {0: 98, 1: 68, 2: 58, 3: 102, 4: 72, 5: 58, 6: 104, 7: 84, 8: 92, 9: 98, 10: 82, 11: 78}

NORMAL DAY 2 (M103_2026_02_19_15_30):
  Directions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
  Trials per direction: {0: 26, 1: 48, 2: 52, 3: 58, 4: 26, 5: 54, 6: 46, 7: 38, 8: 47, 9: 34, 10: 50, 11: 54}

MUSCIMOL (M103_2026_02_20_16_00):
  Directions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
  Trials per direction: {0: 84, 1: 78, 2: 76, 3: 68, 4: 90, 5: 80, 6: 96, 7: 90, 8: 82, 9: 74, 10: 100, 11: 82}




In [ ]:
from tools.params import Params
def plot_single_trial_trace(direction, condition, trial_num, bodypart, pre_ms, post_ms):
    """Plot a single trial's kinematic trace with latency markers."""
    if current_comparator is None:
        print("⚠ No active comparison")
        return
    
    try:
        # Select the appropriate dataset
        if condition == 'Normal Day 1':
            dataset = current_comparator.day1_dataset
        elif condition == 'Normal Day 2':
            dataset = current_comparator.day2_dataset
        else:  # Muscimol
            dataset = current_comparator.muscimol_dataset
        
        # Get kinematics and align
        kin_concat = dataset.get_kinematics(bodypart, direction, 'trial')
        aligned, perturb_idx = dataset.align_to_perturbation(
            kin_concat, direction, pre_ms, post_ms
        )
        
        # Check trial number validity
        if trial_num >= aligned.shape[0]:
            print(f"✗ Trial {trial_num} exceeds available trials (max: {aligned.shape[0]-1})")
            return
        
        # Get single trial
        trial_kin = aligned[trial_num, :, :]  # (n_frames, 3)
        
        # Compute baseline (from -200 to 0ms)
        baseline_start = int((-200 / 1000) / Params.BIN_SIZE + perturb_idx)
        baseline_end = int((0 / 1000) / Params.BIN_SIZE + perturb_idx)
        baseline_start = max(0, baseline_start)
        baseline_end = min(trial_kin.shape[0], baseline_end)
        baseline_pos = trial_kin[baseline_start:baseline_end, :].mean(axis=0)
        
        # Compute metrics for this trial to get latencies
        metrics = current_comparator._compute_single_trial_metrics(
            trial_kin, perturb_idx, (-200, 0), bodypart
        )
        
        # Create time axis in ms relative to perturbation
        n_frames = trial_kin.shape[0]
        time_ms = np.arange(n_frames) * Params.BIN_SIZE * 1000 - (perturb_idx * Params.BIN_SIZE * 1000)
        
        # Compute 3D distance from baseline
        distance_from_baseline = np.linalg.norm(trial_kin - baseline_pos, axis=1)
        
        # Also compute individual components
        x_displacement = trial_kin[:, 0] - baseline_pos[0]
        y_displacement = trial_kin[:, 1] - baseline_pos[1]
        z_displacement = trial_kin[:, 2] - baseline_pos[2]
        
        # Create figure with 2 subplots
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
        
        # Plot 1: 3D distance from baseline
        ax1.plot(time_ms, distance_from_baseline, 'k-', linewidth=2, label='3D Distance from Baseline')
        ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax1.axvline(0, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Perturbation Onset')
        ax1.axvspan(0, 500, alpha=0.1, color='red', label='Solenoid Active (0-500ms)')
        
        # Mark latencies
        colors_latency = {'peak': 'blue', 'recovery': 'green', 'correction': 'orange'}
        
        if metrics['peak_latency_ms'] is not None:
            ax1.axvline(metrics['peak_latency_ms'], color=colors_latency['peak'], 
                        linestyle='-', linewidth=2.5, alpha=0.8, label=f'Peak Latency: {metrics["peak_latency_ms"]:.0f}ms')
        
        if metrics['recovery_latency_ms'] is not None:
            ax1.axvline(metrics['recovery_latency_ms'], color=colors_latency['recovery'], 
                        linestyle='-', linewidth=2.5, alpha=0.8, label=f'Recovery Latency: {metrics["recovery_latency_ms"]:.0f}ms')
        
        if metrics['correction_latency_ms'] is not None:
            ax1.axvline(metrics['correction_latency_ms'], color=colors_latency['correction'], 
                        linestyle='-', linewidth=2.5, alpha=0.8, label=f'Correction Latency: {metrics["correction_latency_ms"]:.0f}ms')
        
        ax1.set_ylabel('Distance from Baseline (cm)', fontsize=11, fontweight='bold')
        ax1.set_title(f"{condition} | Direction {direction} | Trial {trial_num} | {bodypart.replace('_', ' ').upper()}", 
                     fontsize=12, fontweight='bold')
        ax1.grid(True, alpha=0.3)
        ax1.legend(loc='upper left', fontsize=9, framealpha=0.95)
        ax1.set_xlim(time_ms[0], time_ms[-1])
        
        # Plot 2: XYZ components
        ax2.plot(time_ms, x_displacement, label='X (anterior-posterior)', linewidth=2, alpha=0.8)
        ax2.plot(time_ms, y_displacement, label='Y (medial-lateral)', linewidth=2, alpha=0.8)
        ax2.plot(time_ms, z_displacement, label='Z (dorsal-ventral)', linewidth=2, alpha=0.8)
        
        ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax2.axvline(0, color='red', linestyle='--', alpha=0.7, linewidth=2)
        ax2.axvspan(0, 500, alpha=0.1, color='red')
        
        # Mark latencies on xyz plot too
        if metrics['peak_latency_ms'] is not None:
            ax2.axvline(metrics['peak_latency_ms'], color=colors_latency['peak'], 
                        linestyle='-', linewidth=2.5, alpha=0.8)
        if metrics['recovery_latency_ms'] is not None:
            ax2.axvline(metrics['recovery_latency_ms'], color=colors_latency['recovery'], 
                        linestyle='-', linewidth=2.5, alpha=0.8)
        if metrics['correction_latency_ms'] is not None:
            ax2.axvline(metrics['correction_latency_ms'], color=colors_latency['correction'], 
                        linestyle='-', linewidth=2.5, alpha=0.8)
        
        ax2.set_xlabel('Time relative to perturbation onset (ms)', fontsize=11, fontweight='bold')
        ax2.set_ylabel('Displacement from Baseline (cm)', fontsize=11, fontweight='bold')
        ax2.set_title('XYZ Components', fontsize=12, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        ax2.legend(loc='upper left', fontsize=9, framealpha=0.95)
        ax2.set_xlim(time_ms[0], time_ms[-1])
        
        # Add metrics box
        metrics_text = f"Peak Magnitude: {metrics['peak_magnitude_cm']:.2f} cm\n"
        if metrics['recovery_latency_ms'] is not None:
            metrics_text += f"Recovery Latency: {metrics['recovery_latency_ms']:.0f} ms\n"
        else:
            metrics_text += f"Recovery Latency: N/A\n"
        if metrics['correction_latency_ms'] is not None:
            metrics_text += f"Correction Latency: {metrics['correction_latency_ms']:.0f} ms"
        else:
            metrics_text += f"Correction Latency: N/A"
        
        ax2.text(0.98, 0.97, metrics_text, transform=ax2.transAxes, fontsize=9,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.85),
                family='monospace')
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n{'='*80}")
        print(f"Trial Metrics: {condition} | Direction {direction} | Trial {trial_num}")
        print(f"{'='*80}")
        print(f"Peak Latency: {metrics.get('peak_latency_ms', 'N/A'):.0f} ms")
        print(f"Peak Magnitude: {metrics.get('peak_magnitude_cm', 'N/A'):.2f} cm")
        print(f"Recovery Latency: {metrics.get('recovery_latency_ms', 'N/A'):.0f} ms")
        print(f"Correction Latency: {metrics.get('correction_latency_ms', 'N/A'):.0f} ms")
        print(f"{'='*80}\n")
        
    except Exception as e:
        print(f"✗ Error: {str(e)}")
        import traceback
        traceback.print_exc()

# Create control widgets for single trial visualization
condition_dropdown = Dropdown(
    options=['Normal Day 1', 'Normal Day 2', 'Muscimol'],
    value='Normal Day 1',
    description='Condition:',
    layout={'width': '250px'}
)

trial_num_slider = IntSlider(
    value=0,
    min=0,
    max=10,  # Will update based on selection
    step=1,
    description='Trial #:',
    layout={'width': '300px'}
)

trace_bodypart_dropdown = Dropdown(
    options=['left_paw', 'right_paw', 'left_foot', 'right_foot', 'hip_center', 'shoulder_center'],
    value='left_paw',
    description='Body Part:',
    layout={'width': '250px'}
)

out_trace = interactive(plot_single_trial_trace,
                       direction=direction_slider,
                       condition=condition_dropdown,
                       trial_num=trial_num_slider,
                       bodypart=trace_bodypart_dropdown,
                       pre_ms=pre_ms_slider,
                       post_ms=post_ms_slider)

# Update max trials when condition changes
def update_max_trials(change):
    cond = change['new']
    if current_comparator is None:
        return
    
    if cond == 'Normal Day 1':
        max_trials = current_comparator.day1_dataset.n_trials_per_direction[0]
    elif cond == 'Normal Day 2':
        max_trials = current_comparator.day2_dataset.n_trials_per_direction[0]
    else:
        max_trials = current_comparator.muscimol_dataset.n_trials_per_direction[0]
    
    trial_num_slider.max = max(0, max_trials - 1)

condition_dropdown.observe(update_max_trials, names='value')

display(VBox([
    Label("Single Trial Kinematic Trace with Latency Markers"),
    HBox([pair_dropdown]),
    HBox([direction_slider, condition_dropdown]),
    HBox([trial_num_slider, trace_bodypart_dropdown]),
    HBox([pre_ms_slider, post_ms_slider]),
    out_trace
]))

## Single Trial Kinematic Trace with Latency Markers

Visualize individual trial body endpoint trajectories with peak, recovery, and correction latencies marked.

In [ ]:
def plot_trial_annotation_tool(direction, condition, trial_num, bodypart, 
                                manual_peak_ms, manual_recovery_ms, manual_correction_ms,
                                pre_ms, post_ms):
    """
    Interactive tool to manually annotate recovery and correction points on single trial trace.
    Allows adjusting latency markers via sliders and comparing auto vs manual detection.
    """
    if current_comparator is None:
        print("⚠ No active comparison")
        return
    
    try:
        # Select the appropriate dataset
        if condition == 'Normal Day 1':
            dataset = current_comparator.day1_dataset
        elif condition == 'Normal Day 2':
            dataset = current_comparator.day2_dataset
        else:  # Muscimol
            dataset = current_comparator.muscimol_dataset
        
        # Get kinematics and align
        kin_concat = dataset.get_kinematics(bodypart, direction, 'trial')
        aligned, perturb_idx = dataset.align_to_perturbation(
            kin_concat, direction, pre_ms, post_ms
        )
        
        # Check trial number validity
        if trial_num >= aligned.shape[0]:
            print(f"✗ Trial {trial_num} exceeds available trials (max: {aligned.shape[0]-1})")
            return
        
        # Get single trial
        trial_kin = aligned[trial_num, :, :]  # (n_frames, 3)
        
        # Compute baseline
        baseline_start = int((-200 / 1000) / Params.BIN_SIZE + perturb_idx)
        baseline_end = int((0 / 1000) / Params.BIN_SIZE + perturb_idx)
        baseline_start = max(0, baseline_start)
        baseline_end = min(trial_kin.shape[0], baseline_end)
        baseline_pos = trial_kin[baseline_start:baseline_end, :].mean(axis=0)
        
        # Get auto-detected metrics
        auto_metrics = current_comparator._compute_single_trial_metrics(
            trial_kin, perturb_idx, (-200, 0), bodypart
        )
        
        # Create time axis
        n_frames = trial_kin.shape[0]
        time_ms = np.arange(n_frames) * Params.BIN_SIZE * 1000 - (perturb_idx * Params.BIN_SIZE * 1000)
        distance_from_baseline = np.linalg.norm(trial_kin - baseline_pos, axis=1)
        
        x_displacement = trial_kin[:, 0] - baseline_pos[0]
        y_displacement = trial_kin[:, 1] - baseline_pos[1]
        z_displacement = trial_kin[:, 2] - baseline_pos[2]
        
        # === CREATE FIGURE ===
        fig = plt.figure(figsize=(14, 10))
        gs = fig.add_gridspec(3, 1, height_ratios=[3, 3, 1.5], hspace=0.35)
        ax1 = fig.add_subplot(gs[0])
        ax2 = fig.add_subplot(gs[1])
        ax_legend = fig.add_subplot(gs[2])
        ax_legend.axis('off')
        
        # === PLOT 1: 3D Distance from Baseline ===
        ax1.plot(time_ms, distance_from_baseline, 'k-', linewidth=2.5, label='3D Distance from Baseline', zorder=3)
        ax1.axhline(0, color='gray', linestyle='--', alpha=0.4, linewidth=1)
        ax1.axvline(0, color='red', linestyle='--', alpha=0.5, linewidth=2, label='Perturbation Onset (t=0)')
        ax1.axvspan(0, 500, alpha=0.08, color='red', label='Solenoid Active (0-500ms)')
        
        # Color codes
        auto_color = '#2E86AB'
        manual_color = '#A23B72'
        
        # === AUTO-DETECTED LATENCIES (dashed, lower opacity) ===
        if auto_metrics['peak_latency_ms'] is not None:
            ax1.axvline(auto_metrics['peak_latency_ms'], color=auto_color, linestyle='--', 
                        linewidth=2, alpha=0.5, zorder=2)
            ax1.text(auto_metrics['peak_latency_ms'], ax1.get_ylim()[1] * 0.95, 
                    f"Auto Peak\n{auto_metrics['peak_latency_ms']:.0f}ms",
                    fontsize=8, ha='center', bbox=dict(boxstyle='round,pad=0.3', 
                    facecolor=auto_color, alpha=0.3, edgecolor=auto_color, linewidth=1))
        
        if auto_metrics['recovery_latency_ms'] is not None:
            ax1.axvline(auto_metrics['recovery_latency_ms'], color='#06A77D', linestyle='--', 
                        linewidth=2, alpha=0.5, zorder=2)
            ax1.text(auto_metrics['recovery_latency_ms'], ax1.get_ylim()[1] * 0.85, 
                    f"Auto Recovery\n{auto_metrics['recovery_latency_ms']:.0f}ms",
                    fontsize=8, ha='center', bbox=dict(boxstyle='round,pad=0.3', 
                    facecolor='#06A77D', alpha=0.3, edgecolor='#06A77D', linewidth=1))
        
        if auto_metrics['correction_latency_ms'] is not None:
            ax1.axvline(auto_metrics['correction_latency_ms'], color='#F18F01', linestyle='--', 
                        linewidth=2, alpha=0.5, zorder=2)
            ax1.text(auto_metrics['correction_latency_ms'], ax1.get_ylim()[1] * 0.75, 
                    f"Auto Correction\n{auto_metrics['correction_latency_ms']:.0f}ms",
                    fontsize=8, ha='center', bbox=dict(boxstyle='round,pad=0.3', 
                    facecolor='#F18F01', alpha=0.3, edgecolor='#F18F01', linewidth=1))
        
        # === MANUALLY ANNOTATED LATENCIES (solid, full opacity) ===
        if manual_peak_ms is not None and manual_peak_ms >= time_ms[0]:
            ax1.axvline(manual_peak_ms, color=auto_color, linestyle='-', 
                        linewidth=3, alpha=0.9, zorder=4)
            ax1.scatter([manual_peak_ms], [distance_from_baseline[min(int((manual_peak_ms + perturb_idx * Params.BIN_SIZE * 1000) / (Params.BIN_SIZE * 1000)), len(distance_from_baseline)-1)]], 
                       s=150, color=auto_color, marker='o', edgecolors='black', linewidth=1.5, zorder=5, label=f'Manual Peak: {manual_peak_ms:.0f}ms')
        
        if manual_recovery_ms is not None and manual_recovery_ms >= time_ms[0]:
            ax1.axvline(manual_recovery_ms, color='#06A77D', linestyle='-', 
                        linewidth=3, alpha=0.9, zorder=4)
            ax1.scatter([manual_recovery_ms], [distance_from_baseline[min(int((manual_recovery_ms + perturb_idx * Params.BIN_SIZE * 1000) / (Params.BIN_SIZE * 1000)), len(distance_from_baseline)-1)]], 
                       s=150, color='#06A77D', marker='s', edgecolors='black', linewidth=1.5, zorder=5, label=f'Manual Recovery: {manual_recovery_ms:.0f}ms')
        
        if manual_correction_ms is not None and manual_correction_ms >= time_ms[0]:
            ax1.axvline(manual_correction_ms, color='#F18F01', linestyle='-', 
                        linewidth=3, alpha=0.9, zorder=4)
            ax1.scatter([manual_correction_ms], [distance_from_baseline[min(int((manual_correction_ms + perturb_idx * Params.BIN_SIZE * 1000) / (Params.BIN_SIZE * 1000)), len(distance_from_baseline)-1)]], 
                       s=150, color='#F18F01', marker='^', edgecolors='black', linewidth=1.5, zorder=5, label=f'Manual Correction: {manual_correction_ms:.0f}ms')
        
        ax1.set_ylabel('Distance from Baseline (cm)', fontsize=12, fontweight='bold')
        ax1.set_title(f"{condition} | Direction {direction} | Trial {trial_num} | {bodypart.replace('_', ' ').upper()}\n(Auto = dashed lines, Manual = solid lines)", 
                     fontsize=13, fontweight='bold')
        ax1.grid(True, alpha=0.3, linestyle=':', linewidth=0.8)
        ax1.set_xlim(time_ms[0], time_ms[-1])
        ax1.legend(loc='upper left', fontsize=9, framealpha=0.95, ncol=2)
        
        # === PLOT 2: XYZ Components ===
        ax2.plot(time_ms, x_displacement, label='X (anterior-posterior)', linewidth=2, alpha=0.8, color='#C41E3A')
        ax2.plot(time_ms, y_displacement, label='Y (medial-lateral)', linewidth=2, alpha=0.8, color='#2E8B57')
        ax2.plot(time_ms, z_displacement, label='Z (dorsal-ventral)', linewidth=2, alpha=0.8, color='#4169E1')
        
        ax2.axhline(0, color='gray', linestyle='--', alpha=0.4, linewidth=1)
        ax2.axvline(0, color='red', linestyle='--', alpha=0.5, linewidth=2)
        ax2.axvspan(0, 500, alpha=0.08, color='red')
        
        # Mark manual latencies on XYZ plot
        if manual_peak_ms is not None and manual_peak_ms >= time_ms[0]:
            ax2.axvline(manual_peak_ms, color=auto_color, linestyle='-', linewidth=3, alpha=0.6)
        if manual_recovery_ms is not None and manual_recovery_ms >= time_ms[0]:
            ax2.axvline(manual_recovery_ms, color='#06A77D', linestyle='-', linewidth=3, alpha=0.6)
        if manual_correction_ms is not None and manual_correction_ms >= time_ms[0]:
            ax2.axvline(manual_correction_ms, color='#F18F01', linestyle='-', linewidth=3, alpha=0.6)
        
        ax2.set_xlabel('Time relative to perturbation onset (ms)', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Displacement from Baseline (cm)', fontsize=12, fontweight='bold')
        ax2.set_title('XYZ Components', fontsize=12, fontweight='bold')
        ax2.grid(True, alpha=0.3, linestyle=':', linewidth=0.8)
        ax2.legend(loc='upper left', fontsize=9, framealpha=0.95)
        ax2.set_xlim(time_ms[0], time_ms[-1])
        
        # === LEGEND PANEL: Show metrics comparison ===
        legend_text = "AUTO-DETECTED vs MANUAL ANNOTATION\n"
        legend_text += "="*50 + "\n\n"
        
        legend_text += "PEAK LATENCY:\n"
        if auto_metrics['peak_latency_ms'] is not None:
            legend_text += f"  Auto: {auto_metrics['peak_latency_ms']:.0f} ms\n"
        else:
            legend_text += f"  Auto: N/A\n"
        legend_text += f"  Manual: {manual_peak_ms:.0f if manual_peak_ms else 'Not set'} ms\n"
        
        if manual_peak_ms and auto_metrics['peak_latency_ms']:
            diff = manual_peak_ms - auto_metrics['peak_latency_ms']
            legend_text += f"  Δ: {diff:+.0f} ms\n"
        legend_text += "\n"
        
        legend_text += "RECOVERY LATENCY:\n"
        if auto_metrics['recovery_latency_ms'] is not None:
            legend_text += f"  Auto: {auto_metrics['recovery_latency_ms']:.0f} ms\n"
        else:
            legend_text += f"  Auto: N/A\n"
        legend_text += f"  Manual: {manual_recovery_ms:.0f if manual_recovery_ms else 'Not set'} ms\n"
        
        if manual_recovery_ms and auto_metrics['recovery_latency_ms']:
            diff = manual_recovery_ms - auto_metrics['recovery_latency_ms']
            legend_text += f"  Δ: {diff:+.0f} ms\n"
        legend_text += "\n"
        
        legend_text += "CORRECTION LATENCY:\n"
        if auto_metrics['correction_latency_ms'] is not None:
            legend_text += f"  Auto: {auto_metrics['correction_latency_ms']:.0f} ms\n"
        else:
            legend_text += f"  Auto: N/A\n"
        legend_text += f"  Manual: {manual_correction_ms:.0f if manual_correction_ms else 'Not set'} ms\n"
        
        if manual_correction_ms and auto_metrics['correction_latency_ms']:
            diff = manual_correction_ms - auto_metrics['correction_latency_ms']
            legend_text += f"  Δ: {diff:+.0f} ms\n"
        
        ax_legend.text(0.05, 0.95, legend_text, transform=ax_legend.transAxes, 
                      fontsize=10, verticalalignment='top', family='monospace',
                      bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.95, pad=0.8))
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n{'='*80}")
        print(f"MANUAL ANNOTATION TOOL: {condition} | Direction {direction} | Trial {trial_num}")
        print(f"{'='*80}")
        print(f"Body Part: {bodypart.replace('_', ' ').upper()}")
        print(f"\nAuto-Detected:")
        print(f"  Peak Latency: {auto_metrics['peak_latency_ms']:.0f if auto_metrics['peak_latency_ms'] else 'N/A'} ms")
        print(f"  Recovery Latency: {auto_metrics['recovery_latency_ms']:.0f if auto_metrics['recovery_latency_ms'] else 'N/A'} ms")
        print(f"  Correction Latency: {auto_metrics['correction_latency_ms']:.0f if auto_metrics['correction_latency_ms'] else 'N/A'} ms")
        print(f"\nManual Annotations:")
        print(f"  Peak Latency: {manual_peak_ms:.0f if manual_peak_ms else 'Not set'} ms")
        print(f"  Recovery Latency: {manual_recovery_ms:.0f if manual_recovery_ms else 'Not set'} ms")
        print(f"  Correction Latency: {manual_correction_ms:.0f if manual_correction_ms else 'Not set'} ms")
        print(f"{'='*80}\n")
        
    except Exception as e:
        print(f"✗ Error: {str(e)}")
        import traceback
        traceback.print_exc()

# === ANNOTATION TOOL CONTROLS ===
annotation_condition = Dropdown(
    options=['Normal Day 1', 'Normal Day 2', 'Muscimol'],
    value='Normal Day 1',
    description='Condition:',
    layout={'width': '250px'}
)

annotation_trial = IntSlider(
    value=0,
    min=0,
    max=10,
    step=1,
    description='Trial #:',
    layout={'width': '300px'}
)

annotation_bodypart = Dropdown(
    options=['left_paw', 'right_paw', 'left_foot', 'right_foot', 'hip_center', 'shoulder_center'],
    value='left_paw',
    description='Body Part:',
    layout={'width': '250px'}
)

# Sliders for manual latencies (default ranges based on typical response timings)
peak_latency_slider = IntSlider(
    value=50,
    min=0,
    max=500,
    step=10,
    description='Peak (ms):',
    layout={'width': '350px'}
)

recovery_latency_slider = IntSlider(
    value=300,
    min=0,
    max=2000,
    step=10,
    description='Recovery (ms):',
    layout={'width': '350px'}
)

correction_latency_slider = IntSlider(
    value=150,
    min=0,
    max=1500,
    step=10,
    description='Correction (ms):',
    layout={'width': '350px'}
)

# Update max trials when condition changes
def update_max_trials_annotation(change):
    cond = change['new']
    if current_comparator is None:
        return
    
    if cond == 'Normal Day 1':
        max_trials = current_comparator.day1_dataset.n_trials_per_direction[0]
    elif cond == 'Normal Day 2':
        max_trials = current_comparator.day2_dataset.n_trials_per_direction[0]
    else:
        max_trials = current_comparator.muscimol_dataset.n_trials_per_direction[0]
    
    annotation_trial.max = max(0, max_trials - 1)

annotation_condition.observe(update_max_trials_annotation, names='value')

out_annotation = interactive(plot_trial_annotation_tool,
                            direction=direction_slider,
                            condition=annotation_condition,
                            trial_num=annotation_trial,
                            bodypart=annotation_bodypart,
                            manual_peak_ms=peak_latency_slider,
                            manual_recovery_ms=recovery_latency_slider,
                            manual_correction_ms=correction_latency_slider,
                            pre_ms=pre_ms_slider,
                            post_ms=post_ms_slider)

display(VBox([
    Label("✎ Manual Annotation Tool: Mark Recovery & Correction Points", 
         style=dict(font_size='14px', font_weight='bold')),
    Label("Use the sliders to manually annotate peak, recovery, and correction latencies on the single trial trace."),
    Label("Compare with auto-detected values (dashed lines) using the overlaid plots."),
    HBox([pair_dropdown]),
    HBox([direction_slider, annotation_condition]),
    HBox([annotation_trial, annotation_bodypart]),
    Label("👉 Adjust latency markers below (use sliders):", style=dict(font_weight='bold')),
    HBox([peak_latency_slider]),
    HBox([recovery_latency_slider]),
    HBox([correction_latency_slider]),
    HBox([pre_ms_slider, post_ms_slider]),
    out_annotation
]))


In [ ]:
# === ANNOTATION STORAGE ===
from IPython.display import clear_output

# Global dictionary to store manual annotations
manual_annotations = {}

def save_annotation():
    """Save current manual annotation to the global dictionary."""
    if current_comparator is None:
        print("⚠ No active comparison")
        return
    
    try:
        # Build annotation key
        key = f"{current_triplet_name}|dir{direction_slider.value}|{annotation_condition.value}|trial{annotation_trial.value}|{annotation_bodypart.value}"
        
        # Store annotation
        manual_annotations[key] = {
            'triplet': current_triplet_name,
            'direction': direction_slider.value,
            'condition': annotation_condition.value,
            'trial_num': annotation_trial.value,
            'body_part': annotation_bodypart.value,
            'peak_latency_ms': peak_latency_slider.value,
            'recovery_latency_ms': recovery_latency_slider.value,
            'correction_latency_ms': correction_latency_slider.value,
            'timestamp': pd.Timestamp.now()
        }
        
        print(f"✓ Annotation saved for:")
        print(f"  Condition: {annotation_condition.value}")
        print(f"  Direction: {direction_slider.value}")
        print(f"  Trial: {annotation_trial.value}")
        print(f"  Body Part: {annotation_bodypart.value}")
        print(f"  Peak: {peak_latency_slider.value}ms, Recovery: {recovery_latency_slider.value}ms, Correction: {correction_latency_slider.value}ms")
        print(f"  Timestamp: {manual_annotations[key]['timestamp']}")
        print(f"\nTotal saved annotations: {len(manual_annotations)}")
        
    except Exception as e:
        print(f"✗ Error saving annotation: {str(e)}")

def show_saved_annotations():
    """Display all saved annotations as a table."""
    if not manual_annotations:
        print("⚠ No annotations saved yet")
        return
    
    # Convert to DataFrame
    df = pd.DataFrame(list(manual_annotations.values()))
    
    # Reorder columns
    cols = ['triplet', 'condition', 'direction', 'trial_num', 'body_part', 
            'peak_latency_ms', 'recovery_latency_ms', 'correction_latency_ms', 'timestamp']
    df = df[cols]
    
    print(f"\n{'='*140}")
    print(f"SAVED MANUAL ANNOTATIONS ({len(df)} total)")
    print(f"{'='*140}\n")
    
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    display(df)

def export_annotations(filename='manual_annotations.csv'):
    """Export all saved annotations to CSV file."""
    if not manual_annotations:
        print("⚠ No annotations to export")
        return
    
    df = pd.DataFrame(list(manual_annotations.values()))
    df.to_csv(filename, index=False)
    print(f"✓ Annotations exported to {filename} ({len(df)} rows)")

def clear_annotations(confirm=False):
    """Clear all stored annotations."""
    if confirm:
        manual_annotations.clear()
        print("✓ All annotations cleared")
    else:
        print("⚠ Call with confirm=True to clear all annotations")

# Control buttons
save_button = Button(
    description='💾 Save Annotation',
    button_style='info',
    tooltip='Save current manual annotation'
)
save_button.on_click(lambda x: save_annotation())

show_button = Button(
    description='📋 Show Saved',
    button_style='success',
    tooltip='Display all saved annotations'
)
show_button.on_click(lambda x: show_saved_annotations())

export_button = Button(
    description='💾 Export CSV',
    button_style='warning',
    tooltip='Export annotations to CSV file'
)
export_button.on_click(lambda x: export_annotations())

display(VBox([
    Label("💾 Save & Manage Annotations", style=dict(font_weight='bold', font_size='12px')),
    HBox([save_button, show_button, export_button]),
    Label("After adjusting the latency markers above, click 'Save Annotation' to store them.")
]))

,triplet,condition,direction,trial_num,body_part,peak_latency_ms,recovery_latency_ms,correction_latency_ms,timestamp
0,M103: Day1 (M103-Norm-D1) vs Day2 (M103-Norm-D...,Normal Day 1,0,0,left_paw,90,240,90,2026-04-16 02:04:15.419759


In [ ]:
def analyze_annotation_differences():
    """Analyze differences between manual and auto-detected latencies across all saved annotations."""
    if not manual_annotations:
        print("⚠ No annotations saved yet")
        return
    
    results = []
    
    for key, annotation in manual_annotations.items():
        try:
            # Load the dataset
            if annotation['condition'] == 'Normal Day 1':
                dataset = current_comparator.day1_dataset
            elif annotation['condition'] == 'Normal Day 2':
                dataset = current_comparator.day2_dataset
            else:
                dataset = current_comparator.muscimol_dataset
            
            # Get auto-detected metrics
            kin_concat = dataset.get_kinematics(annotation['body_part'], annotation['direction'], 'trial')
            aligned, perturb_idx = dataset.align_to_perturbation(
                kin_concat, annotation['direction'], pre_ms_slider.value, post_ms_slider.value
            )
            
            trial_kin = aligned[annotation['trial_num'], :, :]
            auto_metrics = current_comparator._compute_single_trial_metrics(
                trial_kin, perturb_idx, (-200, 0), annotation['body_part']
            )
            
            # Compute differences
            peak_diff = None
            recovery_diff = None
            correction_diff = None
            
            if auto_metrics['peak_latency_ms'] is not None:
                peak_diff = annotation['peak_latency_ms'] - auto_metrics['peak_latency_ms']
            
            if auto_metrics['recovery_latency_ms'] is not None:
                recovery_diff = annotation['recovery_latency_ms'] - auto_metrics['recovery_latency_ms']
            
            if auto_metrics['correction_latency_ms'] is not None:
                correction_diff = annotation['correction_latency_ms'] - auto_metrics['correction_latency_ms']
            
            results.append({
                'Condition': annotation['condition'],
                'Direction': annotation['direction'],
                'Trial': annotation['trial_num'],
                'Body Part': annotation['body_part'],
                'Manual Peak (ms)': annotation['peak_latency_ms'],
                'Auto Peak (ms)': auto_metrics['peak_latency_ms'],
                'Peak Δ (ms)': peak_diff,
                'Manual Recovery (ms)': annotation['recovery_latency_ms'],
                'Auto Recovery (ms)': auto_metrics['recovery_latency_ms'],
                'Recovery Δ (ms)': recovery_diff,
                'Manual Correction (ms)': annotation['correction_latency_ms'],
                'Auto Correction (ms)': auto_metrics['correction_latency_ms'],
                'Correction Δ (ms)': correction_diff,
            })
        
        except Exception as e:
            print(f"  ⚠ Skipped {key}: {str(e)[:50]}")
    
    if not results:
        print("✗ No valid annotations to analyze")
        return
    
    df = pd.DataFrame(results)
    
    print(f"\n{'='*180}")
    print(f"ANNOTATION VALIDATION: Manual vs Auto-Detected ({len(df)} trials analyzed)")
    print(f"{'='*180}\n")
    
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    
    display(df.round(1))
    
    # Summary statistics
    print(f"\n{'='*180}")
    print(f"SUMMARY STATISTICS: Δ columns (Manual - Auto)")
    print(f"{'='*180}\n")
    
    summary_stats = {}
    for col in ['Peak Δ (ms)', 'Recovery Δ (ms)', 'Correction Δ (ms)']:
        valid_diffs = df[col].dropna()
        if len(valid_diffs) > 0:
            summary_stats[col] = {
                'Mean': valid_diffs.mean(),
                'Median': valid_diffs.median(),
                'Std': valid_diffs.std(),
                'Min': valid_diffs.min(),
                'Max': valid_diffs.max(),
                'n': len(valid_diffs)
            }
    
    summary_df = pd.DataFrame(summary_stats).T
    display(summary_df.round(1))

analyze_button = Button(
    description='📊 Analyze Differences',
    button_style='success',
    tooltip='Compare manual vs auto-detected latencies'
)
analyze_button.on_click(lambda x: analyze_annotation_differences())

display(VBox([
    Label("📊 Validation & Analysis", style=dict(font_weight='bold', font_size='12px')),
    analyze_button,
    Label("Compare manual annotations with auto-detected values to validate your markings.")
]))

,Condition,Direction,Trial,Body Part,Manual Peak (ms),Auto Peak (ms),Peak Δ (ms),Manual Recovery (ms),Auto Recovery (ms),Recovery Δ (ms),Manual Correction (ms),Auto Correction (ms),Correction Δ (ms)
0,Normal Day 1,0,0,left_paw,90,90.0,0.0,240,240.0,0.0,90,90.0,0.0


,Mean,Median,Std,Min,Max,n
Peak Δ (ms),0.0,0.0,NaN,0.0,0.0,1.0
Recovery Δ (ms),0.0,0.0,NaN,0.0,0.0,1.0
Correction Δ (ms),0.0,0.0,NaN,0.0,0.0,1.0
